# Chapter 24 — Building an Embedding Runtime

**Book alignment:** Embeddings From First Principles, Chapter 24

**Question this notebook isolates:** What would a system look like that treats the
representation layer as something to *measure and govern* rather than assume? This notebook
composes the committed artifacts of Waves 1–5 into one "Observatory session" and shows the
near-but-wrong result caught by the *system*, not the geometry.

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    return json.loads((EXP / wave / "artifacts" / f"{name}.json").read_text())

## 1. Register the models — shape, dimensionality, agreement

In [ ]:
ani = art("wave1", "anisotropy")["models"]
dim = art("wave2", "dimensionality-report")["models"]
sc  = art("wave3", "space-comparison")["pairs"]

print("space registry:")
for mdl in ("minilm-l6", "mpnet-base", "bge-large"):
    print(f"  {mdl:12} origin {ani[mdl]['mean_random_cosine']:.2f}  eff.rank {ani[mdl]['effective_rank']:.0f}"
          f"  TwoNN-ID {dim[mdl]['twonn_id']:.1f}")
print("\nspace_comparison:")
for pair, v in sc.items():
    print(f"  {pair:26} CKA {v['linear_cka']:.2f}  10-NN {v['neighborhood_overlap_at10']:.2f}"
          f"  top1-agree {v['top1_retrieval_agreement']:.2f}")

assert ani["minilm-l6"]["mean_random_cosine"] < 0.1 < 0.3 < ani["bge-large"]["mean_random_cosine"]
assert sc["mpnet-base vs bge-large"]["top1_retrieval_agreement"] < 0.75

## 2. Calibrate a duplicate filter; build a bridge; read its scoped verdict

In [ ]:
cal = art("wave1", "calibration")
rp = art("wave3", "relation-preservation")["pairs"]["minilm-l6 vs mpnet-base"]
print(f"duplicate filter (mpnet): AUC {cal['auc']:.2f}, escalate band {cal['escalate_band_fraction']:.0%} of pairs")
print(f"bridge MiniLM->mpnet: paraphrase-vs-negation gap {rp['paraphrase_vs_negation_native']:+.3f} native"
      f" -> {rp['paraphrase_vs_negation_bridged']:+.3f} bridged")
assert cal["escalate_band_fraction"] > 0.8            # the raw score is a weak decision on its own
assert rp["paraphrase_vs_negation_bridged"] < 0       # this bridge is NOT usable_for relation tasks
print("\nthe runtime carries these as data: a wide escalate band, a polarity-inverting bridge -> usable_for gates")

## 3. The near-but-wrong result is caught by the system, not the geometry

In [ ]:
dw = art("wave1", "distractor-winrate")["by_distractor_type"]
sa = art("wave1", "signal-ablation")["accuracy"]

print("a generic NLI reranker does NOT rescue a near-but-wrong negation passage:")
print(f"  bi-encoder distractor-win-rate for negation : {dw['negation']['bi_encoder_distractor_win_rate']:.1%}")
print(f"  after the NLI reranker                       : {dw['negation']['nli_reranker_distractor_win_rate']:.1%}  (worse)")
print(f"\ngeometric signal bundle (margin/density/rank/hubness) balanced accuracy : {sa['geometric']:.2f}")
print(f"raw score alone                                                          : {sa['score_only']:.2f}")
print(f"geometric + NLI                                                          : {sa['geometric_plus_nli']:.2f}")

assert dw["negation"]["nli_reranker_distractor_win_rate"] > dw["negation"]["bi_encoder_distractor_win_rate"]
assert sa["geometric"] - sa["score_only"] > 0.12           # geometry carries the discriminative signal
assert sa["geometric_plus_nli"] <= sa["geometric"] + 0.01  # a bolted-on NLI head adds nothing
print("\nthe runtime did not FIX the embedding - it carried its known limits as data and let policy act")

## What we earned — and what the book established

Every limit in this book is measurable: the objective's bias, the arbitrary basis, the
effective dimension, the hubs, the near-but-wrong tail, the uncalibrated score, the
incompatible second space, the lossy bridge, the forgetful compression. An Embedding
Observatory composes the per-chapter artifacts into a runtime that carries those
measurements as metadata and enforces the book's principles as invariants — including the
separation of **identity** (exact), **compatibility** (measured), and **usability** (a
scoped policy call).

The book began with *a vector is not meaning*. It ends more precisely:

> **Geometry is evidence about a representation, not permission to use it — and whenever you
> transform a representation, measure what survived before treating it as equivalent.**